# 1. Scanner

In [ ]:
import os 
from dotenv import load_dotenv
from newsapi import NewsApiClient 

secret_file = "../secrets/news_api_key.env"
loaded = load_dotenv(secret_file, override=True)
NEWS_API_KEY = os.getenv("NEWS_API_KEY")

if not loaded:
    raise FileNotFoundError(f"Could not load {secret_file}")

if not NEWS_API_KEY:
    raise ValueError("NEWS_API_KEY not found in environment file")

In [7]:
import requests
from requests import Response
from datetime import datetime, timezone, timedelta
from typing import List, Dict, Any, Optional

class Scanner:
    def __init__(self, api_key: str) -> None:
        self.api_key = api_key
        self.base_url = "https://newsapi.org/v2/everything"
    
    def fetch_news(self, query: str, days_back = 3) -> List[Dict[str, Any]]:
        from_date = (datetime.now(timezone.utc) 
                    - timedelta(days=days_back)).strftime("%Y-%m-%d")

        params = {"q": query,
                  "from": from_date,
                  "sortBy": "publishedAt",
                  "language": "en",
                  "apiKey": self.api_key,
                  "pageSize": 10}
        
        response : Response = requests.get(self.base_url, params=params)
        data : Dict[str, Any] = response.json()

        if data.get("status") != "ok":
            raise Exception(f"NewsAPI error: {data}")
        
        articles: List[Dict[str, Any]] = data.get("articles", [])
        return self.normalize(articles)

    def normalize(self, articles: List[Dict[str, Any]]) -> List[Dict[str, Optional[str]]]:
        normalized : List[Dict[str, Optional[str]]] = []

        for article in articles:
            normalized.append({"title": article.get("title"),
                               "description": article.get("description"),
                               "source": article.get("source", {}).get("name"),
                               "published_at": article.get("publishedAt"),
                               "url": article.get("url")
            })
        
        return normalized

In [ ]:
import textwrap
sss
def print_event(event: Dict[str, Optional[str]]) -> None:
    print("\n" + "-" * 100)
    print("TITLE:")
    print(textwrap.fill(event.get("title", ""), 80))

    print("\nSOURCE:", event.get("source"))
    print("DATE:", event.get("published_at"))

    print("\nDESCRIPTION:")
    print(textwrap.fill(event.get("description", ""), 80))

    print("\nURL:", event.get("url"))

In [9]:
query = "EU AI Act"
scanner : Scanner = Scanner(NEWS_API_KEY)
events : List[Dict[str, Optional[str]]] = scanner.fetch_news(query)

for event in events[:5]:
    print_event(event)


----------------------------------------------------------------------------------------------------
TITLE:
EU rules reining in Big Tech will now target cloud services and AI, regulators
say

SOURCE: The Times of India
DATE: 2026-04-28T17:39:54Z

DESCRIPTION:
The European Union is expanding its Digital Markets Act to cover cloud and
artificial intelligence services. This move aims to ensure fairer competition in
these growing digital sectors. Regulators are examining if major companies like
Amazon and Microsoft sh…

URL: https://economictimes.indiatimes.com/tech/technology/eu-rules-reining-in-big-tech-will-now-target-cloud-services-and-ai-regulators-say/articleshow/130587317.cms

----------------------------------------------------------------------------------------------------
TITLE:
EU rules reining in Big Tech will now target cloud services and AI, regulators
say

SOURCE: CNA
DATE: 2026-04-28T17:06:47Z

DESCRIPTION:
BRUSSELS, April 28 : The European Union plans to turn the focus o

In [ ]:
ssssssssdsadas